In [11]:
import os
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

# ==========================================
# 1. SETUP ENGINE & PHYSICAL DATABASE
# ==========================================
class SecureState(TypedDict):
    payload_id: str
    amount: float
    approved: bool

def security_checkpoint_node(state: SecureState):
    print("🛡️ [Security Node] Scanning transaction for fraud risk...")
    return {"approved": False}

def execution_node(state: SecureState):
    print(f"💰 [Execution Node] CRITICAL: Disbursing ${state['amount']} to client!")
    return {"approved": True}

# Build the layout map
builder = StateGraph(SecureState)
builder.add_node("security", security_checkpoint_node)
builder.add_node("disburse", execution_node)

builder.add_edge(START, "security")
builder.add_edge("security", "disburse")
builder.add_edge("disburse", END)

# Use a physical database file on your Mac hard drive instead of volatile RAM
DB_FILE = "production_vault.db"
# Clean up old runs if the file already exists
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)




In [12]:
# ==========================================
# 2. ENDPOINT A: Simulating the User Click
# ==========================================
def simulate_http_endpoint_user_buy():
    print("🌐 [API Endpoint: /submit-payment] Received request from user.")
    
    # Open connection to the physical database file
    with SqliteSaver.from_conn_string(DB_FILE) as checkpointer:
        app = builder.compile(checkpointer=checkpointer, interrupt_before=["disburse"])
        
        config = {"configurable": {"thread_id": "tx_abhishek_2026"}}
        initial_state = {"payload_id": "REQ-888", "amount": 9500.00, "approved": False}
        
        # This triggers execution and FREEZES at the interrupt barrier
        app.invoke(initial_state, config=config)
        
    print("🔌 [API Endpoint: /submit-payment] HTTP Response 202 Sent. Connection Closed.")


In [14]:
# ==========================================
# 3. ENDPOINT B: Simulating the Admin Action Hours Later
# ==========================================
def simulate_http_endpoint_admin_approve():
    print("\n⏰ ... Hours Pass ... The server went to sleep and woke back up ...\n")
    print("🌐 [API Endpoint: /admin/approve] Manager opened dashboard and clicked Approve.")
    
    # Re-open the physical database connection as a completely fresh action
    with SqliteSaver.from_conn_string(DB_FILE) as checkpointer:
        app = builder.compile(checkpointer=checkpointer, interrupt_before=["disburse"])
        
        # The admin only needs to provide the unique thread tracking ID
        config = {"configurable": {"thread_id": "tx_abhishek_2026"}}
        
        # Resume execution out of the file by passing None
        print("📥 Pulling paused token out of physical SQLite file...")
        app.invoke(None, config=config)
        
    print("🔌 [API Endpoint: /admin/approve] HTTP Response 200 Sent. Transaction Closed.")



In [16]:
# Run the independent cycles
simulate_http_endpoint_user_buy()


🌐 [API Endpoint: /submit-payment] Received request from user.
🛡️ [Security Node] Scanning transaction for fraud risk...
🔌 [API Endpoint: /submit-payment] HTTP Response 202 Sent. Connection Closed.


In [17]:
simulate_http_endpoint_admin_approve()


⏰ ... Hours Pass ... The server went to sleep and woke back up ...

🌐 [API Endpoint: /admin/approve] Manager opened dashboard and clicked Approve.
📥 Pulling paused token out of physical SQLite file...
💰 [Execution Node] CRITICAL: Disbursing $9500.0 to client!
🔌 [API Endpoint: /admin/approve] HTTP Response 200 Sent. Transaction Closed.


In [7]:
import sqlite3
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

# 1. Define a minimal graph for demonstration
class SimpleJobState(TypedDict):
    job_name: str
    status: str

def basic_worker(state: SimpleJobState):
    return {"status": "completed"}

builder = StateGraph(SimpleJobState)
builder.add_node("worker", basic_worker)
builder.add_edge(START, "worker")
builder.add_edge("worker", END)

DB_NAME = "multi_job_vault.db"

# 2. Simulate TWO entirely separate jobs running through the system
with SqliteSaver.from_conn_string(DB_NAME) as checkpointer:
    app = builder.compile(checkpointer=checkpointer)
    
    # JOB 1: Running under Thread Alpha
    config_alpha = {"configurable": {"thread_id": "thread_alpha_111"}}
    app.invoke({"job_name": "Data Processing", "status": "pending"}, config=config_alpha)
    
    # JOB 2: Running under Thread Beta
    config_beta = {"configurable": {"thread_id": "thread_beta_222"}}
    app.invoke({"job_name": "Image Rendering", "status": "pending"}, config=config_beta)

print("💾 Both jobs executed and recorded to the database file.\n")

# =========================================================
# RAW SQL INSPECTION: Let's peek inside LangGraph's tables
# =========================================================
print("🔍 Querying internal sqlite database tables directly...\n")

# Connect using standard Python sqlite3 (completely bypassing LangGraph)
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Query the core checkpoint table columns
cursor.execute("SELECT thread_id, checkpoint_ns, checkpoint_id FROM checkpoints;")
rows = cursor.fetchall()

print(f"{'THREAD ID (The Job)':<25} | {'NAMESPACE':<12} | {'CHECKPOINT ID (The Step Version)'}")
print("-" * 85)
for row in rows:
    print(f"{row[0]:<25} | {row[1]:<12} | {row[2]}")

conn.close()

💾 Both jobs executed and recorded to the database file.

🔍 Querying internal sqlite database tables directly...

THREAD ID (The Job)       | NAMESPACE    | CHECKPOINT ID (The Step Version)
-------------------------------------------------------------------------------------
thread_alpha_111          |              | 1f151d62-4d62-6330-bfff-3655a357f161
thread_alpha_111          |              | 1f151d62-4d64-62f2-8000-3ba74708451a
thread_alpha_111          |              | 1f151d62-4d65-694a-8001-edc6644f3f6a
thread_beta_222           |              | 1f151d62-4d6a-6544-bfff-4651e6f099c4
thread_beta_222           |              | 1f151d62-4d6b-662e-8000-be9309c0e0bb
thread_beta_222           |              | 1f151d62-4d6c-6e02-8001-11eb332fe1c0


In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# ==========================================
# 1. STATE DEFINITION WITH EXPLICIT SIGNAL KEY
# ==========================================
class GuardedOrderState(TypedDict):
    item: str
    admin_action: str # Can be "PENDING", "APPROVED", or "REJECTED"
    status: str

# ==========================================
# 2. NODES & ROUTERS
# ==========================================
def entry_processor(state: GuardedOrderState):
    print("📦 [System] Order logged. Freezing for admin review...")
    return {"status": "awaiting_review"}

def payment_execution_node(state: GuardedOrderState):
    print("💳 [Payment Engine] SUCCESS: Capital captured and settled!")
    return {"status": "order_shipped"}

def cancellation_node(state: GuardedOrderState):
    print("🗑️ [System] Order safely cancelled and resource released.")
    return {"status": "order_cancelled"}

def admin_approval_router(state: GuardedOrderState):
    """
    APPLICATION LAYER CHECK: 
    This is where your code explicitly checks the human's input signal.
    """
    print(f"🤖 [Router] Evaluating explicit signal: admin_action='{state['admin_action']}'")
    
    if state["admin_action"] == "APPROVED":
        return "charge_user"
    else:
        return "cancel_order"

# ==========================================
# 3. GRAPH CONFIGURATION
# ==========================================
builder = StateGraph(GuardedOrderState)
builder.add_node("processor", entry_processor)
builder.add_node("payment", payment_execution_node)
builder.add_node("cancellation", cancellation_node)

builder.add_edge(START, "processor")

# Draw a fork in the road AFTER the processor node based on the router evaluation
builder.add_conditional_edges(
    "processor",
    admin_approval_router,
    {
        "charge_user": "payment",
        "cancel_order": "cancellation"
    }
)

builder.add_edge("payment", END)
builder.add_edge("cancellation", END)

memory_db = MemorySaver()

# We pause BEFORE the router evaluates the state data
app = builder.compile(
    checkpointer=memory_db,
    interrupt_before=["payment", "cancellation"] # Halt right after 'processor' runs
)
print("🔒 Secure Conditional Interrupt Graph Compiled!")

# ==========================================
# 4. SIMULATION: USER CLICKS BUY
# ==========================================
print("\n=== STEP 1: User places order ===")
config = {"configurable": {"thread_id": "order_abhishek_2026"}}
initial_payload = {"item": "Quantum_Laptop", "admin_action": "PENDING", "status": "initiated"}

app.invoke(initial_payload, config=config)

# ==========================================
# 5. INSPECTION LAYER: Checking where we stand
# ==========================================
snapshot = app.get_state(config)
print(f"\n🔍 [DB Inspection] Is graph currently paused? {len(snapshot.next) > 0}")
print(f"🔍 [DB Inspection] Next potential nodes to run: {snapshot.next}")
print(f"🔍 [DB Inspection] Current State Data: {snapshot.values}")

# ==========================================
# 6. APPLICATION LAYER: Admin Clicks "REJECT"
# ==========================================
print("\n=== STEP 2: Admin reviews dashboard and clicks 'REJECT' ===")

# We explicitly update the database state state BEFORE waking up the graph engine
print("✍️ Updating database checkpoint values explicitly...")
app.update_state(
    config, 
    {"admin_action": "REJECTED"}, # Overwrite the 'PENDING' string with 'REJECTED'
    as_node="processor"           # Pretend this update came from the processor node
)

print("📥 Sending framework wake-up signal (Passing None)...")
final_state = app.invoke(None, config=config)

print(f"\n🏁 Final System Status: '{final_state['status']}'")

🔒 Secure Conditional Interrupt Graph Compiled!

=== STEP 1: User places order ===
📦 [System] Order logged. Freezing for admin review...
🤖 [Router] Evaluating explicit signal: admin_action='PENDING'

🔍 [DB Inspection] Is graph currently paused? True
🔍 [DB Inspection] Next potential nodes to run: ('cancellation',)
🔍 [DB Inspection] Current State Data: {'item': 'Quantum_Laptop', 'admin_action': 'PENDING', 'status': 'awaiting_review'}

=== STEP 2: Admin reviews dashboard and clicks 'REJECT' ===
✍️ Updating database checkpoint values explicitly...
🤖 [Router] Evaluating explicit signal: admin_action='REJECTED'
📥 Sending framework wake-up signal (Passing None)...
🗑️ [System] Order safely cancelled and resource released.

🏁 Final System Status: 'order_cancelled'
